<a href="https://colab.research.google.com/github/binhminh276/hcmc-house-price-prediction-2stage/blob/main/notebooks/07_Association_Rules.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules


df = pd.read_csv('processed_dataset_cleaned_new.csv', sep=';')

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:

cols_to_float = ['Diện tích', 'Khoảng giá', 'Giá m2']
for col in cols_to_float:
    df[col] = df[col].astype(str).str.replace(',', '.').astype(float)


df_ar = df.drop(columns=['ID', 'Ngày hết hạn', 'Giá m2'])


df_ar['Diện tích'] = pd.qcut(df_ar['Diện tích'], q=3, labels=['Diện tích nhỏ', 'Diện tích vừa', 'Diện tích lớn'])
df_ar['Khoảng giá'] = pd.qcut(df_ar['Khoảng giá'], q=3, labels=['Giá rẻ', 'Giá trung bình', 'Giá cao'])


df_ar['Số phòng ngủ'] = df_ar['Số phòng ngủ'].apply(lambda x: f'{x} PN')
df_ar['Số phòng tắm, vệ sinh'] = df_ar['Số phòng tắm, vệ sinh'].apply(lambda x: f'{x} WC')

binary_cols = ['Mặt tiền', 'Gần bệnh viện', 'Gần chợ', 'Gần trường học', 'Cao tầng', 'Quy hoạch']
for col in binary_cols:
    df_ar[col] = df_ar[col].apply(lambda x: f'Có {col.lower()}' if x == 1 else f'Không {col.lower()}')


df_dummy = pd.get_dummies(df_ar)


danh_sach_support = [0.03, 0.05, 0.08, 0.10, 0.12]
danh_sach_lift = [1.2, 1.5, 2.0, 2.5]
price_targets = ['Khoảng giá_Giá rẻ', 'Khoảng giá_Giá trung bình', 'Khoảng giá_Giá cao']

ket_qua = []
print("⏳ Đang chạy thử nghiệm tự động (Phiên bản chỉ đếm Luật về Giá)...\n")

for supp in danh_sach_support:

    frequent_itemsets = apriori(df_dummy, min_support=supp, use_colnames=True)

    for lift in danh_sach_lift:
        if len(frequent_itemsets) == 0:
            ket_qua.append({'Support': supp, 'Lift': lift, 'Số luật Targe-Price': 0})
            continue


        rules = association_rules(frequent_itemsets, metric="lift", min_threshold=lift)

        if len(rules) == 0:
            ket_qua.append({'Support': supp, 'Lift': lift, 'Số luật Targe-Price': 0})
            continue


        luat_ve_gia = rules[rules['consequents'].apply(lambda y: len(y) == 1 and any(t in y for t in price_targets))]

        ket_qua.append({'Support': supp, 'Lift': lift, 'Số luật Targe-Price': len(luat_ve_gia)})


df_ket_qua = pd.DataFrame(ket_qua)
bang_ma_tran = df_ket_qua.pivot(index='Support', columns='Lift', values='Số luật Targe-Price')

print("=========================================================")
print(" BẢNG MA TRẬN SỐ LƯỢNG LUẬT VỀ 'GIÁ' TẠO RA THEO THAM SỐ ")
print("=========================================================")
print(bang_ma_tran)
print("=========================================================")


⏳ Đang chạy thử nghiệm tự động (Phiên bản chỉ đếm Luật về Giá)...

 BẢNG MA TRẬN SỐ LƯỢNG LUẬT VỀ 'GIÁ' TẠO RA THEO THAM SỐ 
Lift      1.2   1.5   2.0  2.5
Support                       
0.03     5067  3346  1809  660
0.05     1715  1051   475  136
0.08      537   324   138   38
0.10      271   161    82   20
0.12      151    83    52   14


In [ ]:



frequent_itemsets = apriori(df_dummy, min_support=0.08, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.2)


price_targets = ['Khoảng giá_Giá rẻ', 'Khoảng giá_Giá trung bình', 'Khoảng giá_Giá cao']

print("="*85)
print(f"{'BÁO CÁO YẾU TỐ ĐẶC TRƯNG CỦA TỪNG PHÂN KHÚC GIÁ':^85}")
print("="*85)

for target in price_targets:

    filtered_rules = rules[rules['consequents'].apply(lambda y: len(y) == 1 and target in y)]


    top_rules = filtered_rules.sort_values(by='lift', ascending=False).head(5)


    target_name = target.split('_')[1].upper()
    total_rules = len(filtered_rules)

    print(f"\n{'*'*85}")
    print(f"📌 PHÂN TÍCH: PHÂN KHÚC {target_name} (Có {total_rules} luật thỏa mãn)")
    print(f"{'*'*85}")

    if total_rules == 0:
        print(f"  Không tìm thấy luật mạnh nào cho phân khúc {target_name}.")
    else:

        for index, row in top_rules.reset_index().iterrows():

            X_str = ', '.join(list(row['antecedents']))
            Y_str = ', '.join(list(row['consequents']))

            supp = row['support']
            conf = row['confidence']
            lift = row['lift']

            print(f"Luật #{index + 1}:  X ➔ Y")
            print(f"  [X] Tập điều kiện : {X_str}")
            print(f"  [Y] Tập hệ quả    : {Y_str}")
            print(f"  --- Các chỉ số đánh giá ---")
            print(f"  ▪ Support    = {supp:.3f}  | (X và Y cùng xuất hiện trong {supp*100:.1f}% tổng dataset)")
            print(f"  ▪ Confidence = {conf:.3f}  | (Khi có X, xác suất xuất hiện Y là {conf*100:.1f}%)")
            print(f"  ▪ Lift       = {lift:.3f}  | (Sự xuất hiện của X làm tăng khả năng có Y lên {lift:.2f} lần)")
            print("-" * 85)

                   BÁO CÁO YẾU TỐ ĐẶC TRƯNG CỦA TỪNG PHÂN KHÚC GIÁ                   

*************************************************************************************
📌 PHÂN TÍCH: PHÂN KHÚC GIÁ RẺ (Có 210 luật thỏa mãn)
*************************************************************************************
Luật #1:  X ➔ Y
  [X] Tập điều kiện : Diện tích_Diện tích nhỏ, Số phòng ngủ_2 PN, Mặt tiền_Không mặt tiền
  [Y] Tập hệ quả    : Khoảng giá_Giá rẻ
  --- Các chỉ số đánh giá ---
  ▪ Support    = 0.082  | (X và Y cùng xuất hiện trong 8.2% tổng dataset)
  ▪ Confidence = 0.861  | (Khi có X, xác suất xuất hiện Y là 86.1%)
  ▪ Lift       = 2.547  | (Sự xuất hiện của X làm tăng khả năng có Y lên 2.55 lần)
-------------------------------------------------------------------------------------
Luật #2:  X ➔ Y
  [X] Tập điều kiện : Diện tích_Diện tích nhỏ, Gần bệnh viện_Không gần bệnh viện, Số phòng ngủ_2 PN
  [Y] Tập hệ quả    : Khoảng giá_Giá rẻ
  --- Các chỉ số đánh giá ---
  ▪ Support    